# 02 — Scatter / Gather on Flash Attention KV Caches

This notebook adapts the scatter/gather operations from notebook 01 to the
**Flash Attention 2** KV cache layout used on NVIDIA GPUs.

Notebook 01 validated the gather-compress-scatter primitive on the classic
paged attention layout (5D keys, 4D values). On NVIDIA, vLLM defaults to
Flash Attention 2, which uses a simpler layout — both keys and values are
stored as `[num_blocks, block_size, num_kv_heads, head_size]`. The scatter
kernel changes from `reshape_and_cache` to `reshape_and_cache_flash`.

The gather direction becomes trivially simple: `cache[block_idx, offset]`
directly yields `[num_kv_heads, head_size]` — no 5D unpacking needed.

We run the same two experiments as notebook 01:
1. **Round-trip correctness** — scatter → gather → re-scatter is lossless
2. **Per-head compaction** — gather → select per head → scatter back produces
   correct compacted caches

These operations are the foundation for integrating KeyDiff compression
into vLLM on NVIDIA hardware.

## Imports and Setup

In [ ]:
import itertools

import torch
from vllm import _custom_ops as ops

torch.set_grad_enabled(False)

assert torch.cuda.is_available(), "CUDA GPU required"
print(f"GPU: {torch.cuda.get_device_name(0)}")

## Configuration

In [ ]:
DTYPE = torch.float16
NUM_KV_HEADS = 4
HEAD_SIZE = 128
BLOCK_SIZE = 16
SEQ_LEN = 137  # intentionally not a multiple of BLOCK_SIZE
DEVICE = "cuda"

## Flash Attention KV Cache Layout

Flash Attention 2 uses a simpler cache layout than classic paged attention:

- **Key cache:** `[num_blocks, block_size, num_kv_heads, head_size]`
- **Value cache:** `[num_blocks, block_size, num_kv_heads, head_size]`

Both caches have the same shape — unlike the classic layout where keys
use a packed 5D format for memory coalescing.

In vLLM, the combined KV cache is stored as
`[2, num_blocks, block_size, num_kv_heads, head_size]` and split via
`kv_cache.unbind(0)` into separate key and value views. The scatter
kernel is `reshape_and_cache_flash` (instead of `reshape_and_cache`).

This layout matches what vLLM's `FlashAttentionImpl` uses on NVIDIA GPUs
(see `vllm/v1/attention/backends/flash_attn.py`).

In [ ]:
def create_kv_caches_flash(num_blocks, block_size, num_kv_heads, head_size,
                           dtype, device="cuda"):
    """Create KV caches in the Flash Attention 2 layout."""
    key_cache = torch.randn(
        num_blocks, block_size, num_kv_heads, head_size,
        dtype=dtype, device=device,
    )
    value_cache = torch.randn(
        num_blocks, block_size, num_kv_heads, head_size,
        dtype=dtype, device=device,
    )
    return key_cache, value_cache


num_blocks = (SEQ_LEN + BLOCK_SIZE - 1) // BLOCK_SIZE + 4
key_cache, value_cache = create_kv_caches_flash(
    num_blocks, BLOCK_SIZE, NUM_KV_HEADS, HEAD_SIZE, DTYPE, DEVICE,
)

print(f"Key cache shape:   {key_cache.shape}")
print(f"Value cache shape: {value_cache.shape}")
print(f"Blocks allocated:  {num_blocks}")

## Scatter / Gather Functions

The same four functions as notebook 01, adapted for the flash layout:

1. **`build_slot_mapping_for_positions`** — unchanged (layout-agnostic)
2. **`gather_from_paged_cache`** — simplified: direct 2D indexing on
   `[num_blocks, block_size, ...]`, no 5D unpacking
3. **`select_per_head`** — unchanged (operates on dense tensors)
4. **`compact_kv_cache`** — uses `reshape_and_cache_flash` for scatter

In [ ]:
def build_slot_mapping_for_positions(block_table, positions, block_size):
    """Map token positions to flat slot indices in the paged cache.

    Layout-agnostic: works for both classic and flash attention caches.
    """
    logical_block_indices = positions // block_size
    physical_block_indices = block_table[logical_block_indices]
    offsets = positions % block_size
    return physical_block_indices * block_size + offsets


def gather_from_paged_cache(key_cache, value_cache, slot_mapping, block_size):
    """Read tokens from the paged cache into dense
    [num_tokens, num_kv_heads, head_size] tensors.

    For the flash layout [num_blocks, block_size, num_kv_heads, head_size],
    this is direct 2D indexing — no reshape or unpacking needed.
    """
    block_indices = slot_mapping // block_size
    offsets = slot_mapping % block_size

    keys = key_cache[block_indices, offsets]
    values = value_cache[block_indices, offsets]

    return keys, values


def select_per_head(dense, kept_indices):
    """Select tokens per head from a dense
    [seq_len, num_kv_heads, head_size] tensor.

    kept_indices: [num_kv_heads, compacted_len] — which positions to keep
    per head. Returns: [compacted_len, num_kv_heads, head_size]
    """
    head_size = dense.shape[2]
    by_head = dense.permute(1, 0, 2)
    idx = kept_indices.unsqueeze(-1).expand(-1, -1, head_size)
    kept = by_head.gather(1, idx)
    return kept.permute(1, 0, 2).contiguous()


def compact_kv_cache(key_cache, value_cache, slot_mapping, kept_indices,
                     block_table, block_size,
                     kv_cache_dtype="auto", k_scale=None, v_scale=None):
    """Compact a sequence's KV cache using per-head token selection.

    Gathers the full cached context, selects the kept tokens per head,
    and scatters the compacted result back into the first
    compacted_len slots. Uses reshape_and_cache_flash for scatter.
    """
    device = key_cache.device
    compacted_len = kept_indices.shape[1]

    if k_scale is None:
        k_scale = torch.tensor(1.0, dtype=torch.float32, device=device)
    if v_scale is None:
        v_scale = torch.tensor(1.0, dtype=torch.float32, device=device)

    keys_dense, _ = gather_from_paged_cache(
        key_cache, value_cache, slot_mapping, block_size,
    )
    keys_compact = select_per_head(keys_dense, kept_indices)
    del keys_dense

    _, values_dense = gather_from_paged_cache(
        key_cache, value_cache, slot_mapping, block_size,
    )
    values_compact = select_per_head(values_dense, kept_indices)
    del values_dense

    compact_positions = torch.arange(
        compacted_len, dtype=torch.long, device=device,
    )
    compact_slot_mapping = build_slot_mapping_for_positions(
        block_table, compact_positions, block_size,
    )
    ops.reshape_and_cache_flash(
        keys_compact, values_compact,
        key_cache, value_cache,
        compact_slot_mapping, kv_cache_dtype, k_scale, v_scale,
    )

    return compacted_len


print("Functions defined")

## Experiment 1 — Scatter → Gather → Scatter Round Trip

Same invariant as notebook 01: data scattered into the cache can be
gathered back identically, and re-scattered without changing the cache.

With the flash layout this is simpler — no 5D packing to worry about.
The gather is just `cache[block_idx, offset]`.

In [ ]:
torch.manual_seed(42)
keys_original = torch.randn(
    SEQ_LEN, NUM_KV_HEADS, HEAD_SIZE, dtype=DTYPE, device=DEVICE,
)
values_original = torch.randn(
    SEQ_LEN, NUM_KV_HEADS, HEAD_SIZE, dtype=DTYPE, device=DEVICE,
)

num_seq_blocks = (SEQ_LEN + BLOCK_SIZE - 1) // BLOCK_SIZE
block_table = torch.arange(num_seq_blocks, dtype=torch.long, device=DEVICE)

positions = torch.arange(SEQ_LEN, dtype=torch.long, device=DEVICE)
slot_mapping = build_slot_mapping_for_positions(
    block_table, positions, BLOCK_SIZE,
)

print(f"Sequence length:   {SEQ_LEN}")
print(f"Blocks used:       {num_seq_blocks}")
print(f"Slot mapping:      [{slot_mapping[0].item()}, "
      f"{slot_mapping[1].item()}, ..., {slot_mapping[-1].item()}]")
print(f"Keys shape:        {keys_original.shape}")
print(f"Values shape:      {values_original.shape}")

In [ ]:
k_scale = torch.tensor(1.0, dtype=torch.float32, device=DEVICE)
v_scale = torch.tensor(1.0, dtype=torch.float32, device=DEVICE)

ops.reshape_and_cache_flash(
    keys_original, values_original,
    key_cache, value_cache,
    slot_mapping, "auto", k_scale, v_scale,
)
key_cache_snapshot = key_cache.clone()
value_cache_snapshot = value_cache.clone()

print("Scattered original data into cache")

In [ ]:
keys_gathered, values_gathered = gather_from_paged_cache(
    key_cache, value_cache, slot_mapping, BLOCK_SIZE,
)

torch.testing.assert_close(keys_gathered, keys_original, atol=0, rtol=0)
torch.testing.assert_close(values_gathered, values_original, atol=0, rtol=0)

print(f"Gathered keys match original  "
      f"(max diff: {(keys_gathered - keys_original).abs().max().item()})")
print(f"Gathered values match original "
      f"(max diff: {(values_gathered - values_original).abs().max().item()})")

In [ ]:
ops.reshape_and_cache_flash(
    keys_gathered, values_gathered,
    key_cache, value_cache,
    slot_mapping, "auto", k_scale, v_scale,
)

torch.testing.assert_close(
    key_cache, key_cache_snapshot, atol=0, rtol=0,
)
torch.testing.assert_close(
    value_cache, value_cache_snapshot, atol=0, rtol=0,
)

print("Cache unchanged after round trip — scatter/gather is lossless")

## Experiment 2 — Compact with Per-Head Token Selection

Same test as notebook 01: gather all cached tokens, select a random
per-head subset (simulating KeyDiff scoring), scatter the survivors
back into the first `compacted_len` slots.

The `select_per_head` function is identical — it operates on dense
tensors regardless of cache layout. Only the gather and scatter
functions differ.

In [ ]:
COMPRESSION_RATIO = 0.5
compacted_len = int(SEQ_LEN * (1 - COMPRESSION_RATIO))

key_cache, value_cache = create_kv_caches_flash(
    num_blocks, BLOCK_SIZE, NUM_KV_HEADS, HEAD_SIZE, DTYPE, DEVICE,
)
ops.reshape_and_cache_flash(
    keys_original, values_original,
    key_cache, value_cache,
    slot_mapping, "auto", k_scale, v_scale,
)

torch.manual_seed(42)
kept_indices = torch.stack([
    torch.randperm(SEQ_LEN, device=DEVICE)[:compacted_len].sort().values
    for _ in range(NUM_KV_HEADS)
])  # [NUM_KV_HEADS, compacted_len]

expected_keys = select_per_head(keys_original, kept_indices)
expected_values = select_per_head(values_original, kept_indices)

print(f"Original sequence:  {SEQ_LEN} tokens")
print(f"Compression ratio:  {COMPRESSION_RATIO}")
print(f"Compacted length:   {compacted_len} tokens")
print(f"Kept indices shape: {kept_indices.shape}")
print(f"Expected keys:      {expected_keys.shape}")

In [ ]:
result_len = compact_kv_cache(
    key_cache, value_cache,
    slot_mapping, kept_indices, block_table,
    BLOCK_SIZE,
    kv_cache_dtype="auto", k_scale=k_scale, v_scale=v_scale,
)

assert result_len == compacted_len

compact_positions = torch.arange(
    compacted_len, dtype=torch.long, device=DEVICE,
)
compact_slot_mapping = build_slot_mapping_for_positions(
    block_table, compact_positions, BLOCK_SIZE,
)
keys_after, values_after = gather_from_paged_cache(
    key_cache, value_cache, compact_slot_mapping, BLOCK_SIZE,
)

torch.testing.assert_close(keys_after, expected_keys, atol=0, rtol=0)
torch.testing.assert_close(values_after, expected_values, atol=0, rtol=0)

print(f"Compacted {SEQ_LEN} -> {compacted_len} tokens")
print(f"Keys match expected selection   "
      f"(max diff: {(keys_after - expected_keys).abs().max().item()})")
print(f"Values match expected selection  "
      f"(max diff: {(values_after - expected_values).abs().max().item()})")

## Parametric Validation

Run both tests across multiple configurations to verify the operations
hold generally. Same parameter sweep as notebook 01.

In [ ]:
DTYPES = [torch.bfloat16, torch.float16]
KV_HEADS = [4, 8]
HEAD_SIZES = [64, 128]
BLOCK_SIZES = [16, 32]
COMP_RATIOS = [0.25, 0.5]

passed = 0
failed = 0

for dtype, nkv, hs, bs in itertools.product(
    DTYPES, KV_HEADS, HEAD_SIZES, BLOCK_SIZES,
):
    seq_len = 137
    nb = (seq_len + bs - 1) // bs + 4

    kc, vc = create_kv_caches_flash(nb, bs, nkv, hs, dtype, DEVICE)

    keys_orig = torch.randn(seq_len, nkv, hs, dtype=dtype, device=DEVICE)
    vals_orig = torch.randn(seq_len, nkv, hs, dtype=dtype, device=DEVICE)

    nsb = (seq_len + bs - 1) // bs
    bt = torch.arange(nsb, dtype=torch.long, device=DEVICE)
    pos = torch.arange(seq_len, dtype=torch.long, device=DEVICE)
    sm = build_slot_mapping_for_positions(bt, pos, bs)

    ks = torch.tensor(1.0, dtype=torch.float32, device=DEVICE)
    vs = torch.tensor(1.0, dtype=torch.float32, device=DEVICE)

    ops.reshape_and_cache_flash(keys_orig, vals_orig, kc, vc, sm, "auto", ks, vs)

    # Round-trip test
    kg, vg = gather_from_paged_cache(kc, vc, sm, bs)
    try:
        torch.testing.assert_close(kg, keys_orig, atol=0, rtol=0)
        torch.testing.assert_close(vg, vals_orig, atol=0, rtol=0)
        passed += 1
    except Exception as e:
        failed += 1
        print(f"FAIL round-trip: dtype={dtype}, heads={nkv}, "
              f"hs={hs}, bs={bs}: {e}")

    # Compact tests
    for cr in COMP_RATIOS:
        cl = int(seq_len * (1 - cr))

        kc2, vc2 = create_kv_caches_flash(nb, bs, nkv, hs, dtype, DEVICE)
        ops.reshape_and_cache_flash(
            keys_orig, vals_orig, kc2, vc2, sm, "auto", ks, vs,
        )

        ki = torch.stack([
            torch.randperm(seq_len, device=DEVICE)[:cl].sort().values
            for _ in range(nkv)
        ])

        ek = select_per_head(keys_orig, ki)
        ev = select_per_head(vals_orig, ki)

        rl = compact_kv_cache(
            kc2, vc2, sm, ki, bt, bs,
            kv_cache_dtype="auto", k_scale=ks, v_scale=vs,
        )

        cp = torch.arange(cl, dtype=torch.long, device=DEVICE)
        csm = build_slot_mapping_for_positions(bt, cp, bs)
        ka, va = gather_from_paged_cache(kc2, vc2, csm, bs)

        try:
            assert rl == cl
            torch.testing.assert_close(ka, ek, atol=0, rtol=0)
            torch.testing.assert_close(va, ev, atol=0, rtol=0)
            passed += 1
        except Exception as e:
            failed += 1
            print(f"FAIL compact: dtype={dtype}, heads={nkv}, "
                  f"hs={hs}, bs={bs}, cr={cr}: {e}")

total = passed + failed
print(f"\n{passed}/{total} tests passed")
if failed == 0:
    print("All configurations verified")

## Notes and Next Steps

**Layout verified.** The scatter/gather and compaction primitives work
correctly on the Flash Attention 2 cache layout
`[num_blocks, block_size, num_kv_heads, head_size]`, using
`reshape_and_cache_flash` for scatter. This completes step 3 of the
integration roadmap.

**Simpler than classic.** The flash layout eliminates the 5D key packing
that made the classic-layout gather complex. The gather function is now
two lines of direct indexing — no reshape or unpacking needed.

**What this enables:** With both cache layouts validated, the next step
is to replace random token selection with KeyDiff's actual key-similarity
scoring (step 4 in the roadmap). This means importing the scoring logic
from kvpress and applying it to the gathered dense keys, then validating
that compacted caches produce correct attention outputs end-to-end.